# 데모 1 — Workflow → Loop Engineering

**한 줄 요약**: 같은 질문 세트가 s1 → s4 를 거치며 답이 좋아지는 과정을 눈으로 확인한다.

| 단계 | 이름 | 구조 | 핵심 질문 |
|---|---|---|---|
| s1 | 직선 파이프라인 | 함수 2개 | 검색이 빗나가면? → **그대로 오답** |
| s2 | 같은 로직을 그래프로 | 노드 2개, 직선 엣지 | 뭐가 좋아졌나? → **답은 그대로, 구조가 보이기 시작** |
| s3 | Agentic RAG | +관련성 평가, +쿼리 재작성 → **루프** | 빗나간 검색을 스스로 복구할 수 있나? |
| s4 | Adaptive RAG | +질문 라우터 (3분기) | 모든 질문이 같은 문을 지나야 하나? |

```mermaid
flowchart LR
    s1["s1 직선<br/>retrieve→generate"] --> s2["s2 그래프化<br/>구조 가시화"]
    s2 --> s3["s3 루프<br/>평가→재작성→재검색"]
    s3 --> s4["s4 라우터<br/>질문별 다른 경로"]
    style s3 fill:#fff3bf
    style s4 fill:#d3f9d8
```

**실행법**: 위에서부터 셀을 하나씩 실행한다. `LLM_PROVIDER=mock`(기본)이면
API 키·네트워크 없이 항상 같은 결과가 재현된다. 실 모델은 `.env`에서 `openai|ollama`로 변경.

In [ ]:
# ── 환경 점검 + 오늘의 질문 세트 ─────────────────────────────
from common.llm import provider
from common.console import banner, step, show_answer, diff_panel, compare_table, stream_run, console
from common.retriever import load_corpus, get_retriever, format_docs
from common.viz import show_graph

QUESTIONS = {
    "easy":   "LangGraph에서 조건부 엣지는 무엇이고 언제 쓰나요?",          # 코퍼스에 정답 있음
    "miss":   "LangGraph로 human approval 단계를 넣으려면 어떻게 해요?",   # 어휘 불일치 → 검색이 빗나감
    "web":    "LangGraph 최신 릴리스 소식을 웹에서 검색해 요약해줘",        # 코퍼스 밖(최신 정보)
    "direct": "안녕하세요! 데모 시작에 앞서 인사드립니다",                  # 검색 불필요
}
results = {}  # (단계, 질문키) → 답변 수집 → 마지막에 before/after 총괄표

banner("데모 1 — Workflow → Loop Engineering", f"LLM_PROVIDER = {provider()}")
console.print("코퍼스(내부 용어집):", [d.metadata["title"] for d in load_corpus()])

---
## s1 — 직선 파이프라인 (LangGraph 없이)

가장 흔한 RAG 의 시작점: **함수 2개를 이어 붙인 직선**이다.

```mermaid
flowchart LR
    Q([질문]) --> R["retrieve(q)<br/>코퍼스 검색"] --> G["generate(q, docs)<br/>LLM 생성"] --> A([답변])
```

판단 지점이 없다. 검색이 무엇을 물어오든 생성은 그대로 진행된다.

In [ ]:
# ── s1 구현: 순수 함수 2개가 전부 ────────────────────────────
from common.llm import get_llm

# '허술한' 생성 프롬프트(v1) — 컨텍스트에 근거가 없어도 답하게 방치한다.
# (데모 3에서 이 프롬프트를 v2로 개선하는 실험을 한다)
# 프롬프트 규약: 지시문을 앞에, [질문] 섹션을 마지막에 둔다
GENERATION_PROMPT = (
    "다음 컨텍스트를 참고해 질문에 한국어로 간결하게 답하라.\n"
    "[컨텍스트]\n{context}\n[질문]\n{question}"
)

retriever = get_retriever(k=2)

def retrieve(question: str):
    """코퍼스에서 관련 문서 top-2 검색."""
    return retriever.retrieve(question)

def generate(question: str, docs) -> str:
    """검색 결과를 근거로 답변 생성."""
    prompt = GENERATION_PROMPT.format(context=format_docs(docs), question=question)
    return get_llm("generator").invoke(prompt).content

def s1_pipeline(question: str) -> str:
    docs = retrieve(question)                      # ① 검색
    console.print("  검색된 문서:", [d.metadata["source"] for d in docs])
    return generate(question, docs)                # ② 생성 — 검색 품질과 무관하게 무조건 실행

In [ ]:
# ── s1 실행 ①: 코퍼스에 정답이 있는 질문 → 잘 답한다 ─────────
step("s1 × 정상 질문")
ans = s1_pipeline(QUESTIONS["easy"])
results[("s1", "easy")] = ans
show_answer(QUESTIONS["easy"], ans)

In [ ]:
# ── s1 실행 ②: 어휘가 어긋난 질문 → 빗나간 검색을 그대로 사용 ──
step("s1 × 한계 질문 (영어 섞인 구어체)")
ans = s1_pipeline(QUESTIONS["miss"])
results[("s1", "miss")] = ans
show_answer(QUESTIONS["miss"], ans, title="답변 (문제 있음)")

**관찰**: 질문은 HITL(사람 승인)을 물었는데, 검색은 `human` 이라는 단어 하나에 걸린
엉뚱한 문서를 가져왔고, 생성은 그 문서를 근거로 **그럴듯한 동문서답**을 만들었다.

> s1 의 한계: *검색이 빗나가도 파이프라인은 멈추지도, 되묻지도 않는다.*
> 실패가 감지되지 않고 그대로 사용자에게 전달된다.

고치기 전에 먼저 할 일이 있다 — **구조를 보이게 만드는 것**. 그래야 어디에
판단 지점을 끼워 넣을지 말할 수 있다.

---
## s2 — 같은 로직을 LangGraph StateGraph 로

로직은 한 줄도 바꾸지 않는다. 대신 **상태(State)·노드·엣지**로 구조를 선언한다.

```mermaid
flowchart LR
    START(("START")) --> R[retrieve] --> G[generate] --> END_(("END"))
```

얻는 것: ① 그래프 시각화 ② 노드 단위 스트리밍/관측 ③ 다음 단계에서 노드·엣지를
"끼워 넣을 수 있는" 자리. 잃는 것: 없음(코드 몇 줄).

In [ ]:
# ── s2 구현: 상태 선언 + 노드 2개 + 직선 엣지 ────────────────
from typing import List, TypedDict
from langchain_core.documents import Document
from langgraph.graph import StateGraph, START, END

class RagState(TypedDict, total=False):
    question: str
    documents: List[Document]
    answer: str

def retrieve_node(state: RagState) -> dict:
    return {"documents": retrieve(state["question"])}     # s1 함수 재사용

def generate_node(state: RagState) -> dict:
    return {"answer": generate(state["question"], state.get("documents", []))}

g2 = StateGraph(RagState)
g2.add_node("retrieve", retrieve_node)
g2.add_node("generate", generate_node)
g2.add_edge(START, "retrieve")
g2.add_edge("retrieve", "generate")
g2.add_edge("generate", END)
app_s2 = g2.compile()
print("s2 그래프 컴파일 완료 — 노드 2개, 직선 엣지")

In [ ]:
# ── s2 그래프 시각화 (outputs/graph_s2.png 저장) ─────────────
show_graph(app_s2, "s2")

In [ ]:
# ── s2 실행: 답은 s1과 완전히 동일하다 (그게 포인트) ──────────
step("s2 × 두 질문")
for key in ("easy", "miss"):
    final, _, _ = stream_run(app_s2, {"question": QUESTIONS[key]})
    results[("s2", key)] = final["answer"]

diff_panel(
    [
        "로직 변화 없음 — 답변은 s1과 글자까지 동일 (아래 표로 확인)",
        "대신 실행이 노드 단위로 보이기 시작했다 (스트리밍 로그, 그래프 그림)",
        "다음 단계부터 이 구조 '사이'에 판단 지점을 끼워 넣는다",
    ],
    title="s1 → s2 에서 달라진 점",
)
compare_table(
    "s1 vs s2 — 같은 질문, 같은 답 (구조만 달라짐)",
    ["질문", "s1 답변(직선 함수)", "s2 답변(그래프)"],
    [
        [QUESTIONS[k][:26] + "…", results[("s1", k)][:44] + "…", results[("s2", k)][:44] + "…"]
        for k in ("easy", "miss")
    ],
)

---
## s3 — 루프의 탄생: 평가하고, 다시 묻는다 (Agentic RAG)

s1/s2 의 병은 "빗나간 검색을 **감지하지 못하는 것**"이었다. 두 노드를 끼워 넣는다.

- `grade_documents`: 문서별로 "질문에 쓸모 있나?" yes/no 판정 → 쓸모없는 문서 제거
- `transform_query`: 관련 문서가 0개면 질문을 표준 용어로 재작성 → **retrieve 로 되돌아간다**

```mermaid
flowchart LR
    START(("START")) --> R[retrieve] --> GD[grade_documents]
    GD -- "관련 문서 있음" --> G[generate] --> END_(("END"))
    GD -. "없음 & 재작성 ≤ 2" .-> T[transform_query] -.-> R
    GD -- "없음 & 예산 소진" --> G
    style T fill:#fff3bf
```

되돌아가는 점선 엣지가 이 데모의 주인공 — **그래프에 처음 생긴 루프**다.
그리고 루프에는 반드시 **탈출 조건**(`rewrite_count ≤ 2`)이 필요하다.
탈출 조건 없는 루프는 기능이 아니라 버그다.

In [ ]:
# ── s3 구현: 새 노드 2개 + 조건부 엣지 + 탈출 카운터 ─────────
MAX_REWRITES = 2

GRADER_PROMPT = (
    "당신은 문서 관련성 평가자다. 문서가 질문에 답하는 근거로 쓸모 있으면 yes,\n"
    "아니면 no 만 출력하라.\n[질문]\n{question}\n[문서]\n{document}"
)
REWRITER_PROMPT = (
    "당신은 검색 쿼리 재작성기다. 아래 질문을 내부 용어집이 쓰는 표준 용어로\n"
    "바꿔 검색 쿼리 한 줄만 출력하라.\n[질문]\n{question}"
)

class RagStateV3(RagState, total=False):
    rewrite_count: int          # ← 루프 탈출 조건용 카운터 (상태에 명시)

def grade_documents_node(state: RagStateV3) -> dict:
    grader = get_llm("grader")
    keep = []
    for d in state["documents"]:
        verdict = grader.invoke(
            GRADER_PROMPT.format(question=state["question"], document=d.page_content)
        ).content.strip().lower()
        console.print(f"    · 관련성 판정 {d.metadata['source']} → [bold]{verdict}[/]")
        if verdict.startswith("y"):
            keep.append(d)
    return {"documents": keep}

def transform_query_node(state: RagStateV3) -> dict:
    new_q = get_llm("rewriter").invoke(
        REWRITER_PROMPT.format(question=state["question"])
    ).content.strip()
    console.print(f"    · 쿼리 재작성: [italic]{new_q}[/]")
    return {"question": new_q, "rewrite_count": state.get("rewrite_count", 0) + 1}

def decide_to_generate(state: RagStateV3) -> str:
    """조건부 엣지 판정: 문서 있음 → 생성 / 없음 → 재작성 (예산 소진 시 탈출)."""
    if state["documents"]:
        return "generate"
    if state.get("rewrite_count", 0) >= MAX_REWRITES:
        console.print(f"    · 재작성 예산 소진(max {MAX_REWRITES}) → 탈출하여 생성으로")
        return "generate"
    return "transform_query"

g3 = StateGraph(RagStateV3)
g3.add_node("retrieve", retrieve_node)            # s2 그대로
g3.add_node("generate", generate_node)            # s2 그대로
g3.add_node("grade_documents", grade_documents_node)   # + 추가
g3.add_node("transform_query", transform_query_node)   # + 추가
g3.add_edge(START, "retrieve")
g3.add_edge("retrieve", "grade_documents")
g3.add_conditional_edges(                          # + 조건부 엣지 (분기)
    "grade_documents", decide_to_generate,
    {"generate": "generate", "transform_query": "transform_query"},
)
g3.add_edge("transform_query", "retrieve")         # + 되돌아가는 엣지 = 루프!
g3.add_edge("generate", END)
app_s3 = g3.compile()
print("s3 그래프 컴파일 완료 — 루프 있는 그래프")

In [ ]:
show_graph(app_s3, "s3")

In [ ]:
# ── s3 실행: s1/s2가 틀렸던 바로 그 질문 ─────────────────────
step("s3 × 한계 질문 — 루프가 도는 것을 관찰")
final, _, miss_path = stream_run(app_s3, {"question": QUESTIONS["miss"], "rewrite_count": 0})
results[("s3", "miss")] = final["answer"]
console.print(f"\n실행 경로: [bold]{' → '.join(miss_path)}[/]")
assert miss_path.count("retrieve") >= 2 and "transform_query" in miss_path, "루프가 돌지 않았습니다!"
show_answer(QUESTIONS["miss"], final["answer"], title="답변 (루프가 복구함)")

In [ ]:
# ── 정상 질문은? 루프 없이 한 번에 통과 (비용 증가 없음) ──────
step("s3 × 정상 질문 — 루프는 필요할 때만 돈다")
final, _, path = stream_run(app_s3, {"question": QUESTIONS["easy"], "rewrite_count": 0})
results[("s3", "easy")] = final["answer"]
console.print(f"\n실행 경로: [bold]{' → '.join(path)}[/]")

compare_table(
    "s2 vs s3 — 한계 질문의 before / after",
    ["", "s2 (직선)", "s3 (루프)"],
    [
        ["실행 경로", "retrieve → generate", " → ".join(miss_path)],
        ["답변", results[("s2", "miss")][:60] + "…", results[("s3", "miss")][:60] + "…"],
    ],
)
diff_panel(
    [
        "+ grade_documents: 빗나간 검색을 '감지'한다",
        "+ transform_query → retrieve: 감지된 실패를 '복구'한다 (루프)",
        "+ rewrite_count 탈출 조건: 루프는 반드시 끝나야 한다 (max 2회)",
    ],
    title="s2 → s3 에서 달라진 점",
)

---
## s4 — 문 앞의 라우터: 질문마다 다른 경로 (Adaptive RAG)

남은 문제: 인사말도, 최신 뉴스 질문도 전부 코퍼스 검색으로 들어간다.
진입점에 **라우터**를 세워 질문 성격대로 보낸다.

```mermaid
flowchart LR
    START(("START")) --> RT{route_question}
    RT -- vectorstore --> R[retrieve] --> GD[grade_documents]
    GD -- yes --> G[generate] --> END_(("END"))
    GD -. no .-> T[transform_query] -.-> R
    RT -- web_search --> W[web_search] --> G
    RT -- direct --> D[direct_answer] --> END_
    style RT fill:#d0ebff
```

In [ ]:
# ── s4 구현: 라우터 + 웹 검색 + 직접 답변 노드 추가 ──────────
from common.websearch import web_search, format_web_results

ROUTER_PROMPT = (
    "당신은 질문 라우터다. 아래 질문을 보고 다음 중 하나만 소문자로 출력하라.\n"
    "- vectorstore: 내부 용어집(LangGraph/Langfuse/에이전트 설계)으로 답할 수 있는 질문\n"
    "- web_search: 최신 소식·외부 정보가 필요한 질문\n"
    "- direct: 검색이 필요 없는 인사·잡담·단순 질문\n[질문]\n{question}"
)
DIRECT_PROMPT = "검색 없이 바로 한국어로 간결하게 답하라.\n[질문]\n{question}"

class RagStateV4(RagStateV3, total=False):
    route: str

def route_question_node(state: RagStateV4) -> dict:
    label = get_llm("router").invoke(
        ROUTER_PROMPT.format(question=state["question"])
    ).content.strip().lower()
    if label not in ("vectorstore", "web_search", "direct"):
        label = "vectorstore"                     # 안전망
    return {"route": label, "rewrite_count": state.get("rewrite_count", 0)}

def web_search_node(state: RagStateV4) -> dict:
    docs = [Document(
        page_content=format_web_results(web_search(state["question"])),
        metadata={"source": "web_search", "title": "웹 검색 결과"},
    )]
    return {"documents": docs}

def direct_answer_node(state: RagStateV4) -> dict:
    return {"answer": get_llm("direct").invoke(
        DIRECT_PROMPT.format(question=state["question"])
    ).content}

g4 = StateGraph(RagStateV4)
for name, fn in [
    ("route_question", route_question_node),      # + 추가
    ("retrieve", retrieve_node),
    ("grade_documents", grade_documents_node),
    ("transform_query", transform_query_node),
    ("web_search", web_search_node),              # + 추가
    ("generate", generate_node),
    ("direct_answer", direct_answer_node),        # + 추가
]:
    g4.add_node(name, fn)
g4.add_edge(START, "route_question")
g4.add_conditional_edges("route_question", lambda s: s["route"], {
    "vectorstore": "retrieve", "web_search": "web_search", "direct": "direct_answer",
})
g4.add_edge("retrieve", "grade_documents")
g4.add_conditional_edges("grade_documents", decide_to_generate,
                         {"generate": "generate", "transform_query": "transform_query"})
g4.add_edge("transform_query", "retrieve")
g4.add_edge("web_search", "generate")
g4.add_edge("generate", END)
g4.add_edge("direct_answer", END)
app_s4 = g4.compile()
print("s4 그래프 컴파일 완료 — 완성형 Adaptive RAG")

In [ ]:
show_graph(app_s4, "s4")

In [ ]:
# ── s4 실행: 4개 질문이 각자 다른 경로로 ─────────────────────
for key in ("easy", "miss", "web", "direct"):
    step(f"s4 × {key}")
    final, _, path = stream_run(app_s4, {"question": QUESTIONS[key], "rewrite_count": 0})
    results[("s4", key)] = final["answer"]
    console.print(f"  경로: [bold]{' → '.join(path)}[/]\n")

> **재사용 안내**: 이 s4 그래프는 `demo1_workflow_to_loop/adaptive_rag.py` 의
> `build_adaptive_rag()` 로 모듈화되어 있다 (노트북과 동일 구현).
> 데모 2(HITL)·데모 3(관측)·데모 5(딥 에이전트)가 이 팩토리를 임포트해 재사용한다.

In [ ]:
# ── 총괄: 단계 × 질문 before/after 매트릭스 ──────────────────
snip = lambda s: (s.replace("\n", " ")[:42] + "…") if s and len(s) > 42 else (s or "—")
rows = []
for key, label in [("easy", "정상"), ("miss", "한계(어휘 불일치)"), ("web", "최신 정보"), ("direct", "인사")]:
    rows.append([
        f"{label}\n{QUESTIONS[key][:22]}…",
        snip(results.get(("s1", key))), snip(results.get(("s2", key))),
        snip(results.get(("s3", key))), snip(results.get(("s4", key))),
    ])
compare_table("단계별 답변 변화 총괄 (— 는 해당 단계에서 미실행)",
              ["질문", "s1 직선", "s2 그래프", "s3 루프", "s4 라우터"], rows)

## 마무리 — 오늘의 3줄

1. **s1→s2**: 로직이 아니라 *구조*를 먼저 바꿨다. 보이지 않으면 고칠 수 없다.
2. **s2→s3**: 판단 지점(관련성 평가)과 되돌아가는 엣지 하나로 시스템이 *자기 수정*을 시작했다.
   단, 루프에는 반드시 탈출 조건.
3. **s3→s4**: 라우터로 질문마다 경로가 달라졌다 — 이제 "실행 경로 자체"가 데이터다.
   경로가 다양해지는 순간 **관측(데모 3)** 이 필요해진다.

**다음 데모 예고**: 이 그래프의 `generate` 뒤에 "발행" 같은 위험한 행동이 붙는다면?
→ 데모 2: Human-in-the-Loop (`interrupt`)